In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re

In [0]:
def _pivot_xml_name_value_pairs(df):
    """
    Pivot XML name-value pair arrays into proper columns.

    Detects columns of type array<struct<_VALUE_, _name_>> — a common XML
    pattern where each <column name="X">v</column> element becomes one
    element of the array — and pivots them into wide format so that each
    unique _name becomes its own column with _VALUE_ as the value.

    If no such columns are found, the DataFrame is returned unchanged.
    """
    nv_columns = []
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType):
            elem = field.dataType.elementType
            if isinstance(elem, StructType):
                elem_field_names = {f.name for f in elem.fields}
                if {"_VALUE_", "_name"}.issubset(elem_field_names):
                    nv_columns.append(field.name)

    if not nv_columns:
        return df  # No name-value pair arrays found

    print(f"[PIVOT] Detected name-value pair columns: {nv_columns}")

    # Add a monotonic row identifier so we can group back after exploding
    df = df.withColumn("_pivot_row_id", F.monotonically_increasing_id())

    # Columns that are NOT name-value arrays (to preserve alongside pivot)
    other_cols = [
        f.name for f in df.schema.fields
        if f.name not in nv_columns and f.name != "_pivot_row_id"
    ]

    for nv_col in nv_columns:
        # Explode the name-value array
        exploded = df.select(
            "_pivot_row_id", *other_cols,
            F.explode_outer(F.col(f"{nv_col}")).alias("_nv_struct")
        )

        # Extract the name and value from each struct
        pairs = exploded.select(
            "_pivot_row_id", *other_cols,
            F.col("_nv_struct._name").alias("_col_name"),
            F.col("_nv_struct._VALUE_").alias("_col_value")
        )

        # Pivot: each unique _name becomes a column, _VALUE becomes the value
        pivoted = (
            pairs
            .groupBy("_pivot_row_id", *other_cols)
            .pivot("_col_name")
            .agg(F.first("_col_value"))
        )

        # Replace the original DataFrame with the pivoted one
        df = pivoted

        # Drop helper column
        df = df.drop("_pivot_row_id")

        print(f"[PIVOT] Pivoted into {len(df.columns)} columns")

        return df


In [0]:
def _flatten_nested_df(df, separator="_"):
    """
    Recursively flatten a DataFrame with nested structs and arrays.

    - StructType columns are expanded into individual columns
      (e.g. address.city -> address_city).
    - ArrayType columns are exploded using explode_outer to preserve
      rows with null/empty arrays.
    - The process repeats until no complex types remain.
    """

    def _has_complex_fields(schema):
        return any(
            isinstance(f.dataType, (StructType, ArrayType))
            for f in schema.fields
        )

    while _has_complex_fields(df.schema):
        for field in df.schema.fields:
            col_name = field.name

            if isinstance(field.dataType, StructType):
                # Expand each nested field as a top-level column
                expanded = [
                    F.col(f"{col_name}.{sub.name}").alias(f"{col_name}{separator}{sub.name}")
                    for sub in field.dataType.fields
                ]
                df = df.select(
                    *[F.col(f"{c.name}") for c in df.schema.fields if c.name != col_name],
                    *expanded
                )
                break  # Restart loop — schema has changed

            elif isinstance(field.dataType, ArrayType):
                # Explode the array; use explode_outer to keep nulls
                df = df.withColumn(col_name, F.explode_outer(F.col(f"{col_name}")))
                break  # Restart loop — element may itself be a struct

    # Clean column names: replace dots, spaces, special characters
    for col_name in df.columns:
        clean_name = (
            col_name
            .replace(".", "_")
            .replace(" ", "_")
            .replace("-", "_")
            .replace("/", "_")
        )
        df = df.withColumnRenamed(col_name, clean_name)

    return df


In [0]:
def _clean_xsi_columns_optimized(df):
    cols = df.columns
    new_cols = []
    drop_cols = set()

    # Map base column = xsi:nil column
    xsi_map = {}
    for c in cols:
        if "xsi:nil" in c:
            base = re.sub(r"_xsi:nil", "", c)
            xsi_map[base] = c

    for c in cols:
        # VALUE column
        if c.endswith("_VALUE"):
            base = c[:-6]

            if base in xsi_map:
                xsi_col = xsi_map[base]
                new_cols.append(
                    when(col(f"{xsi_col}") == True, lit(None))
                    .otherwise(col(f"{c}"))
                    .cast("string")
                    .alias(base)
                )
                drop_cols.update([c, xsi_col])
            else:
                new_cols.append(col(f"{c}").cast("string").alias(base))
                drop_cols.add(c)

        # ONLY xsi:nil (like city)
        elif c in xsi_map.values():
            base = re.sub(r"_xsi:nil", "", c)
            if base not in cols:
                new_cols.append(lit(None).cast("string").alias(base))
            drop_cols.add(c)

        # Normal column
        elif c not in drop_cols:
            new_cols.append(col(f"{c}"))

    return df.select(*new_cols)


In [0]:
df = spark.read.format("xml").options(rowTag="library").load("/Volumes/test_catalog/default/test_volume/xml_file/complex-nested.xml")
df = _pivot_xml_name_value_pairs(df)
df = _flatten_nested_df(df)
df = _clean_xsi_columns_optimized(df)

df.printSchema()
display(df)